<a href="https://colab.research.google.com/github/yohperez/EjemplosMates/blob/main/analisis_imagen_bounding_box.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼️ Detección de objetos en imágenes mediante matrices

## ¿De qué trata este notebook?

Este notebook simula de forma simplificada cómo funciona la **detección de objetos en visión artificial**.

Una imagen en blanco y negro se representa como una **matriz de 10×10** donde:
- `0` = píxel **negro** (fondo)
- `1` = píxel **blanco** (objeto)

El algoritmo recorre la matriz buscando el primer píxel blanco (`1`) y, a partir de él, calcula una **bounding box** (caja delimitadora): las coordenadas y dimensiones del objeto encontrado.

### 🔍 ¿Qué es una bounding box?
Es un rectángulo imaginario que rodea al objeto detectado. Se define con:
- `(x, y)` → esquina superior izquierda del objeto
- `ancho` → número de píxeles blancos consecutivos en esa fila
- `alto` → número de píxeles blancos consecutivos en esa columna

> ⚠️ **Limitación importante**: el algoritmo asume que el objeto es rectangular y mide desde el *primer* píxel blanco encontrado. Esto puede dar resultados incorrectos si el objeto tiene una forma irregular (como un óvalo o diamante).

## 1. Código original

In [ ]:
# Representación de una imagen en blanco y negro como una matriz
# La forma de los 1s dibuja un ÓVALO / DIAMANTE horizontal
imagen = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]

# Función para encontrar un objeto rectangular
def encontrar_objeto(imagen):
    filas = len(imagen)
    columnas = len(imagen[0])

    # Búsqueda de píxeles blancos
    for i in range(filas):
        for j in range(columnas):
            if imagen[i][j] == 1:
                # Se encontró un píxel blanco, buscar el objeto
                ancho = 0
                alto = 0
                for k in range(j, columnas):
                    if imagen[i][k] == 1:
                        ancho += 1
                    else:
                        break
                for k in range(i, filas):
                    if imagen[k][j] == 1:
                        alto += 1
                    else:
                        break

                # Devolver la bounding box
                return (j, i, ancho, alto)

    # No se encontró ningún objeto
    return None

# Buscar el objeto en la imagen
bounding_box = encontrar_objeto(imagen)

# Imprimir los resultados
if bounding_box:
    x, y, ancho, alto = bounding_box
    print("Objeto encontrado en:")
    print(f"  x: {x}, y: {y}")
    print(f"  Ancho: {ancho}, Alto: {alto}")
else:
    print("No se encontró ningún objeto.")

## 2. Visualización de la matriz original

Vamos a visualizar la imagen para confirmar qué forma tiene el objeto.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualizar_imagen(imagen, bounding_box=None, titulo='Imagen'):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(imagen, cmap='gray', vmin=0, vmax=1)
    ax.set_title(titulo, fontsize=14)
    ax.set_xticks(range(len(imagen[0])))
    ax.set_yticks(range(len(imagen)))
    ax.grid(True, color='lightblue', linewidth=0.5)

    if bounding_box:
        x, y, ancho, alto = bounding_box
        rect = patches.Rectangle(
            (x - 0.5, y - 0.5), ancho, alto,
            linewidth=2, edgecolor='red', facecolor='none',
            label=f'Bounding box: x={x}, y={y}, w={ancho}, h={alto}'
        )
        ax.add_patch(rect)
        ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()

bb_original = encontrar_objeto(imagen)
visualizar_imagen(imagen, bb_original, titulo='Imagen original con bounding box')
print(f'Bounding box: {bb_original}')

## 3. Experimento: matriz modificada con un rectángulo perfecto

**Hipótesis**: el algoritmo funciona correctamente cuando el objeto es un **rectángulo**, porque mide el ancho y el alto desde el primer píxel blanco de forma lineal.

Vamos a probarlo con un rectángulo perfecto para confirmar que la bounding box coincide exactamente con el objeto.

In [ ]:
# Matriz modificada: objeto RECTANGULAR perfecto
imagen_rectangulo = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]

bb_rect = encontrar_objeto(imagen_rectangulo)
visualizar_imagen(imagen_rectangulo, bb_rect, titulo='Rectángulo perfecto — bounding box exacta')
print(f'Bounding box rectángulo: {bb_rect}')

## 4. Experimento: objeto con forma irregular (triángulo)

Ahora probamos con una forma irregular para ver cómo falla el algoritmo cuando el objeto **no es rectangular**.

In [ ]:
# Matriz modificada: objeto TRIANGULAR
imagen_triangulo = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
    [0, 0, 1, 1, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]

bb_tri = encontrar_objeto(imagen_triangulo)
visualizar_imagen(imagen_triangulo, bb_tri, titulo='Triángulo — bounding box imprecisa')
print(f'Bounding box triángulo: {bb_tri}')

## 5. ✅ Conclusiones

### ¿Qué hace este código?
Simula un **detector de objetos básico** en imágenes binarias (blanco y negro). La imagen se representa como una matriz 10×10 de ceros y unos, y el algoritmo localiza el primer objeto blanco calculando su posición y tamaño.

### 🔎 ¿Qué forma tiene el objeto original?
El objeto de la matriz original tiene forma de **óvalo o diamante**: las filas centrales son más anchas y se van estrechando hacia arriba y hacia abajo. Esto ya supone un problema para el algoritmo.

### ⚠️ Limitación detectada
El algoritmo mide el `ancho` y el `alto` **desde el primer píxel blanco** encontrado (fila 2, columna 3), contando solo en línea recta. Esto significa:
- El **ancho** medido es `4` (los 4 píxeles de la fila 2), pero las filas centrales tienen **8 píxeles** de ancho.
- El **alto** medido desde la columna 3 es correcto verticalmente, pero la columna 3 no es la más representativa.

**En resumen: el algoritmo solo funciona bien con objetos rectangulares.** Para formas irregulares, la bounding box es imprecisa. Una solución más robusta sería recorrer *todos* los píxeles blancos y calcular los valores `min_x`, `max_x`, `min_y`, `max_y`.

### 🌍 Conexión con la realidad
Este tipo de detección es la base de sistemas reales de **visión por computador** (OpenCV, YOLO, etc.), donde las bounding boxes se usan para detectar caras, coches, animales... La diferencia es que los modelos modernos aprenden a detectar objetos de cualquier forma y tamaño.

---
*Notebook analizado y ampliado a partir del original compartido en Google Colab.*